# Projeto #3: Performance Analysis

## Projeto #3: Performance Analysis

**Domínio:** Esportes / Performance de jogadores
**Pergunta:** Como está o jogador? Vai melhorar? Qual o valor de mercado?
**Conceitos cobertos:** Fundamentos + Prompting (S1-2), RAG histórico (S3),
Tool Use — stats e market data (S5), Agentic Loop streaming (S5-6), LangGraph
pipeline de tempo real (S6-7), Streaming/Real-time (S7), Confidence Scoring
de previsão de mercado (S7), Observability de latência de updates (S8)

**Sobre realismo:** chama a **API real da Anthropic** quando
`ANTHROPIC_API_KEY` está no ambiente, com fallback determinístico sem ela.

In [1]:
!pip install -q langgraph pydantic anthropic pandas

import os
import random
import time
from typing import Optional
from pydantic import BaseModel, Field

### 1. Schema + dados reais — Fantasy Premier League 2023-24

**Por que estas classes existem:** `PlayerSnapshot` é o dado bruto (o que
sabemos objetivamente sobre o jogador agora); `PlayerAnalysis` é a saída do
agente — a interpretação (tendência, previsão de valor, confiança). Separar
os dois deixa claro o que é fato vs o que é inferência do modelo.

Os dados vêm do [Fantasy Premier League dataset](https://github.com/vaastav/Fantasy-Premier-League)
(MIT, mantido por vaastav) — 865 jogadores reais da temporada 2023-24, com
gols, assistências, minutos e `now_cost` (preço real no jogo oficial da
Premier League, usado aqui como proxy de valor de mercado). O dataset não
publica idade dos jogadores, então esse campo fica de fora do
`PlayerSnapshot` na versão real (existia só na versão sintética).

In [2]:
class PlayerSnapshot(BaseModel):
    player_id: str
    games_played: int
    goals: int
    assists: int
    minutes_per_game: float
    market_value_m: float  # em milhões

class PlayerAnalysis(BaseModel):
    player_id: str
    trend: str
    market_value_prediction_m: float
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: str

import pandas as pd

DATASET_PATH = "../../datasets/fpl_player_stats_2023-24.csv"

def load_real_players(path: str = DATASET_PATH, n: int = 12, seed: int = 3) -> list[PlayerSnapshot]:
    df = pd.read_csv(path)
    df = df[df["starts"] > 0]  # só jogadores que efetivamente jogaram
    sample = df.sample(n=n, random_state=seed).reset_index(drop=True)
    players = []
    for _, row in sample.iterrows():
        players.append(PlayerSnapshot(
            player_id=str(row["web_name"]),
            games_played=int(row["starts"]),
            goals=int(row["goals_scored"]),
            assists=int(row["assists"]),
            minutes_per_game=round(float(row["minutes"]) / max(int(row["starts"]), 1), 1),
            market_value_m=round(float(row["now_cost"]) / 10, 1),  # now_cost em décimos de £M
        ))
    return players

def generate_players(n: int = 12, seed: int = 3) -> list[PlayerSnapshot]:
    """Fallback sintético — só usado se o CSV real não for encontrado."""
    random.seed(seed)
    out = []
    for i in range(n):
        out.append(PlayerSnapshot(
            player_id=f"player_{i:03d}",
            games_played=random.randint(5, 38),
            goals=random.randint(0, 25),
            assists=random.randint(0, 15),
            minutes_per_game=round(random.uniform(20, 90), 1),
            market_value_m=round(random.uniform(1, 80), 1),
        ))
    return out

try:
    players = load_real_players()
    print(f"✓ {len(players)} jogadores REAIS carregados (Fantasy Premier League 2023-24)")
except FileNotFoundError:
    players = generate_players()
    print(f"⚠️  datasets/fpl_player_stats_2023-24.csv não encontrado — usando {len(players)} jogadores sintéticos")
print(players[0])

✓ 12 jogadores REAIS carregados (Fantasy Premier League 2023-24)
player_id='N.Semedo' games_played=36 goals=0 assists=2 minutes_per_game=85.7 market_value_m=4.5


**Resultado esperado:** `✓ 12 jogadores REAIS carregados...` e o primeiro
jogador da amostra — um `web_name` real da Premier League 2023-24 (ex.:
"Haaland", "Salah", dependendo do sorteio com seed 3), com gols/assistências
reais daquela temporada.

### 2. RAG histórico (Semana 3) — desempenho passado do mesmo jogador

**Por que esta função existe:** pra avaliar "tendência" (subindo/caindo),
o agente precisa de contexto temporal — não dá pra julgar um jogador só pelo
snapshot atual. `retrieve_historical_seasons` simula a busca de 2 temporadas
anteriores no data warehouse (em produção: uma query real, não RAG
vetorial — o "R" de retrieval aqui é sobre dados estruturados, não texto).

In [3]:
def retrieve_historical_seasons(player: PlayerSnapshot) -> list[dict]:
    """Simula busca de temporadas anteriores (substitua por query real
    ao seu data warehouse de estatísticas)."""
    random.seed(hash(player.player_id) % 1000)
    seasons = []
    for s in range(2):
        seasons.append({
            "season": f"202{3+s}",
            "goals": max(0, player.goals + random.randint(-5, 3)),
            "market_value_m": round(player.market_value_m * random.uniform(0.7, 1.1), 1),
        })
    return seasons

### 3. Tools — stats ao vivo e dados de mercado (Semana 5)

**Por que estas funções existem:** são as duas fontes externas que o agente
consulta a cada "tick" de tempo real (seção 6) — uma API de estatísticas ao
vivo e uma de comparáveis de mercado. Cada uma tem sua própria seed baseada
no `player_id`, então os valores são estáveis entre ticks do mesmo jogador
mas diferentes entre jogadores.

In [4]:
def tool_live_stats(player_id: str) -> dict:
    random.seed(hash(player_id) % 500)
    return {"player_id": player_id, "live_rating": round(random.uniform(5.5, 9.5), 1)}

def tool_market_comparables(player: PlayerSnapshot) -> dict:
    peer_avg = player.market_value_m * random.uniform(0.85, 1.15)
    return {"peer_avg_value_m": round(peer_avg, 1)}

### 4. Chamada ao LLM — real com fallback (Confidence Scoring, Semana 7)

**Por que esta função existe:** é onde o agente decide a tendência e prevê o
valor de mercado. `call_claude_player_analysis` pede pra Claude interpretar
os 4 sinais (histórico, live rating, mercado, jogos disputados) e devolver
uma predição estruturada; `heuristic_player_analysis` calcula a mesma coisa
com regras determinísticas. A parte importante pedagógica é o cálculo de
`confidence`: ele cai quando há poucos dados (`games_played` baixo) OU
quando a previsão diverge muito do valor de mercado atual — confiança não é
um número arbitrário, é uma função de quão bem-fundamentada é a resposta.

In [5]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
USE_REAL_LLM = bool(ANTHROPIC_API_KEY)

if USE_REAL_LLM:
    import anthropic
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

PLAYER_TOOL_SCHEMA = {
    "name": "record_player_analysis",
    "description": "Registra a análise estruturada do jogador",
    "input_schema": {
        "type": "object",
        "properties": {
            "trend": {"type": "string", "enum": ["subindo", "estável", "caindo"]},
            "market_value_prediction_m": {"type": "number"},
            "confidence": {"type": "number", "minimum": 0, "maximum": 1},
            "rationale": {"type": "string"},
        },
        "required": ["trend", "market_value_prediction_m", "confidence", "rationale"],
    },
}

def call_claude_player_analysis(player: PlayerSnapshot, history: list[dict], live: dict, market: dict) -> Optional[dict]:
    if not USE_REAL_LLM:
        return None
    response = _client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        tools=[PLAYER_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "record_player_analysis"},
        messages=[{
            "role": "user",
            "content": (
                f"Jogador com {player.games_played} jogos, {player.goals} gols, "
                f"valor atual ${player.market_value_m}M. Histórico: {history}. "
                f"Live rating: {live['live_rating']}. Média de mercado de pares: "
                f"${market['peer_avg_value_m']}M. Avalie tendência e preveja valor de mercado."
            ),
        }],
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None

def heuristic_player_analysis(player: PlayerSnapshot, history: list[dict], live: dict, market: dict) -> PlayerAnalysis:
    """Fallback — mesmo cálculo de confiança que um prompt real usaria."""
    goal_trend = player.goals - (history[0]["goals"] if history else player.goals)
    trend = "subindo" if goal_trend > 0 else ("estável" if goal_trend == 0 else "caindo")

    predicted_value = (player.market_value_m + market["peer_avg_value_m"]) / 2
    if trend == "subindo":
        predicted_value *= 1.1
    elif trend == "caindo":
        predicted_value *= 0.9

    data_quality = min(player.games_played / 20, 1.0)
    market_agreement = 1 - min(abs(predicted_value - player.market_value_m) / max(player.market_value_m, 1), 1)
    confidence = round(0.5 * data_quality + 0.5 * market_agreement, 2)

    return PlayerAnalysis(
        player_id=player.player_id,
        trend=trend,
        market_value_prediction_m=round(predicted_value, 1),
        confidence=confidence,
        rationale=f"live_rating={live['live_rating']}, peer_avg=${market['peer_avg_value_m']}M, trend={trend}",
    )

def mock_player_analysis(player: PlayerSnapshot, history: list[dict], live: dict, market: dict) -> PlayerAnalysis:
    result = call_claude_player_analysis(player, history, live, market)
    if result is not None:
        return PlayerAnalysis(player_id=player.player_id, **result)
    return heuristic_player_analysis(player, history, live, market)

print(f"🔑 Modo: {'API REAL (Claude Haiku)' if USE_REAL_LLM else 'MOCK — defina ANTHROPIC_API_KEY pra usar a API real'}")

🔑 Modo: MOCK — defina ANTHROPIC_API_KEY pra usar a API real


**Resultado esperado:** `🔑 Modo: MOCK — ...` sem chave configurada.

### 5. Grafo de streaming (Semana 6-7): Ingest → Update → Report, em loop

**Por que esta estrutura existe:** simula um pipeline de tempo real — a cada
"tick" (em produção: um evento do Cloud Pub/Sub) o grafo inteiro roda de
novo com dados frescos. `node_ingest` mede sua própria latência
(`tick_latency_ms`), que é exatamente a métrica que você monitoraria em
produção pra saber se o pipeline está acompanhando o ritmo dos eventos.

In [6]:
from langgraph.graph import StateGraph, START, END

class LiveState(BaseModel):
    player: PlayerSnapshot
    history: list[dict] = []
    live: Optional[dict] = None
    market: Optional[dict] = None
    analysis: Optional[PlayerAnalysis] = None
    tick_latency_ms: float = 0

def node_ingest(state: LiveState) -> LiveState:
    t0 = time.time()
    state.history = retrieve_historical_seasons(state.player)
    state.live = tool_live_stats(state.player.player_id)
    state.market = tool_market_comparables(state.player)
    state.tick_latency_ms = (time.time() - t0) * 1000
    return state

def node_update(state: LiveState) -> LiveState:
    state.analysis = mock_player_analysis(state.player, state.history, state.live, state.market)
    return state

def node_report(state: LiveState) -> LiveState:
    return state

graph = StateGraph(LiveState)
graph.add_node("ingest", node_ingest)
graph.add_node("update", node_update)
graph.add_node("report", node_report)
graph.add_edge(START, "ingest")
graph.add_edge("ingest", "update")
graph.add_edge("update", "report")
graph.add_edge("report", END)

performance_agent = graph.compile()

### 6. Simulando 3 ticks de "tempo real" pra um jogador

In [7]:
player = players[0]
print(f"📡 Streaming updates pra {player.player_id}\n")
for tick in range(3):
    result = performance_agent.invoke(LiveState(player=player))
    a = result["analysis"] if isinstance(result, dict) else result.analysis
    lat = result["tick_latency_ms"] if isinstance(result, dict) else result.tick_latency_ms
    print(f"[tick {tick+1}] trend={a.trend} valor_previsto=${a.market_value_prediction_m}M "
          f"confidence={a.confidence} (ingest_latency={lat:.1f}ms)")
    if a.confidence < 0.5:
        print("   ⚠️  confiança baixa — escalar pra analista humano")
    time.sleep(0.2)  # simula intervalo entre updates

📡 Streaming updates pra N.Semedo

[tick 1] trend=estável valor_previsto=$4.3M confidence=0.98 (ingest_latency=0.1ms)
[tick 2] trend=estável valor_previsto=$4.3M confidence=0.98 (ingest_latency=0.1ms)
[tick 3] trend=estável valor_previsto=$4.3M confidence=0.98 (ingest_latency=0.1ms)


**Resultado esperado:** 3 linhas `[tick N] trend=... valor_previsto=$XM
confidence=0.XX (ingest_latency=~0.0-0.1ms)`. No modo mock, os 3 ticks dão o
mesmo resultado (os dados de entrada não mudam entre ticks nesta simulação
simplificada) — em produção, cada tick traria dados novos do Pub/Sub e a
análise mudaria tick a tick.

### 7. Testes básicos (Semana 9)

In [8]:
def test_confidence_in_range():
    result = performance_agent.invoke(LiveState(player=players[0]))
    a = result["analysis"] if isinstance(result, dict) else result.analysis
    assert 0.0 <= a.confidence <= 1.0
    print("✓ test_confidence_in_range passou")

def test_trend_is_valid_category():
    result = performance_agent.invoke(LiveState(player=players[1]))
    a = result["analysis"] if isinstance(result, dict) else result.analysis
    assert a.trend in {"subindo", "estável", "caindo"}
    print("✓ test_trend_is_valid_category passou")

test_confidence_in_range()
test_trend_is_valid_category()

✓ test_confidence_in_range passou
✓ test_trend_is_valid_category passou


**Resultado esperado:** 2 linhas `✓ ... passou`.

**Próximos passos pra produção:**
- Já dá pra usar Claude de verdade — só definir `ANTHROPIC_API_KEY`
- Trocar o loop de "ticks" por consumo real de Cloud Pub/Sub
- `tool_live_stats` → API real de dados ao vivo (Opta, StatsBomb, etc)
- Avaliação: comparar `market_value_prediction_m` vs valor real de mercado (Transfermarkt)
- Deploy: Cloud Run com endpoint de streaming (Server-Sent Events ou WebSocket)